# w9_flash_ablation.ipynb — @512 STRUCTURE ABLATION (exp/cmp/pool grid)

The full 30-cell structure grid moved out of w9_flash.ipynb (which now holds
only the loss-functional ladder): {no-I, I@dep, I@E, shared-E, dual-E} x
{per-view/pooled CE} x {deployed / exp(128->256->512) / cmp(128->128->64)}.
Same machinery, same shared claims -- safe to run alongside any other pod.
ce@512 and i2ce@512 are DONE and print as references. AUTO-STOPS when drained.


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # shared campaign out dir

# structure grid (arm, cap) -- done cells skip automatically
FLASH = [
    ("wcle_i2ce_icetf", 512),            # I2@dep + per-view CE@dep
    ("wcle_i2expce_icetf", 512),         # I2@dep + per-view CE@E (original)
    ("wcle_i2poolce_icetf", 512),        # I2@dep + pooled CE@dep
    ("wcle_ceexpi2_icetf", 512),         # per-view CE@dep + I2@E
    ("wcle_poolceexpi2_icetf", 512),     # pooled CE@dep + I2@E
    ("wcle_expi2expce_icetf", 512),      # DUAL E: I@E_I + CE@E_CE
    ("wcle_expi2poolexpce_icetf", 512),  # DUAL E: I@E_I + pool->E_CE->CE
    ("wcle_shexpi2ce_icetf", 512),       # SHARED E: I@E + CE@E
    ("wcle_shexpi2poolce_icetf", 512),   # SHARED E: I@E + pool->E->CE
    ("wcle_ce_cetf", 512),               # per-view CE@dep (DONE, skips)
    ("wcle_expce_cetf", 512),            # per-view CE@E
    ("wcle_i2poolexpce_icetf", 512),     # I2@dep + pool->exp->CE
    ("wcle_i2poolcmpce_icetf", 512),     # I2@dep + pool->cmp->CE
    ("wcle_expi2cmpce_icetf", 512),      # DUAL: I2@exp + CE@cmp
    ("wcle_cmpi2expce_icetf", 512),      # DUAL: I2@cmp + CE@exp
    ("wcle_cmpi2cmpce_icetf", 512),      # DUAL: I2@cmp + CE@cmp
    ("wcle_expi2poolcmpce_icetf", 512),  # DUAL: I2@exp + pool->cmp->CE
    ("wcle_shexpi2poolexpce_icetf", 512),  # SHARED exp: pool-BEFORE-E CE
    ("wcle_shcmpi2poolcmpce_icetf", 512),  # SHARED cmp: pool-BEFORE-E CE
    ("wcle_poolcmpce_cetf", 512),        # no-I pool->cmp->CE
    ("wcle_cecmpi2_icetf", 512),         # CE@dep per-view + I2@cmp
    ("wcle_poolcecmpi2_icetf", 512),     # CE@dep pooled + I2@cmp
    ("wcle_shcmpi2poolce_icetf", 512),   # SHARED cmp, pool-AFTER-E
    ("wcle_cmpi2poolexpce_icetf", 512),  # DUAL: I2@cmp + pool->exp->CE
    ("wcle_cmpi2poolcmpce_icetf", 512),  # DUAL: I2@cmp + pool->cmp->CE
    ("wcle_cmpce_cetf", 512),            # no-I CE@cmp
    ("wcle_i2cmpce_icetf", 512),         # I2@dep + CE@cmp
    ("wcle_shcmpi2ce_icetf", 512),       # SHARED cmp: I2@cmp + CE@cmp
    ("wcle_poolce_cetf", 512),           # pooled CE@dep
    ("wcle_poolexpce_cetf", 512),        # pool->E->CE
]
# GPU packing ceiling, INCLUSIVE: a cap packs (two towers/GPU) iff
# cap <= MAXANCHOR. At 2048, the 2048 cells DO pack; only 4096 runs
# solo. Set 4096 to also pack 4096, 1024 to keep 2048 solo, 0 to
# disable packing. Session-local -- a packed crash demotes one rung
# ([512,1024,2048,4096]) for THIS pod only.
MAXANCHOR = 2048
os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(FLASH)} structure cells")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Drain in TWO WAVES with GPU co-residency (ancient vicgrl/vicnogrl
# style; the full-pool mmap is page-cache shared so host RAM stays flat).
# PACK_MAX starts at the notebook constant MAXANCHOR and is SESSION-LOCAL:
# any packed job that crashes demotes the ceiling one rung (user: one
# crash per pod is a fine tuition; a persisted file could be poisoned by
# a RAM-OOM misjudged as VRAM-OOM and wrongly demote the GPU class
# globally). Demoted-out jobs reroute to the solo wave mid-flight.
import queue, subprocess, threading, time
from pathlib import Path

PACK = 2                               # co-resident towers per GPU
RUNGS = [0, 512, 1024, 2048, 4096]
PACK_MAX = MAXANCHOR                   # session-local runtime ceiling
_pmlock = threading.Lock()
print(f"[pack] MAXANCHOR = {MAXANCHOR} (notebook constant, session-local)")

def _demote(cap):
    global PACK_MAX
    with _pmlock:
        new = max(r for r in RUNGS if r < cap)
        if new < PACK_MAX:
            PACK_MAX = new
            print(f"[pack] packed job died at cap {cap} -> "
                  f"session maxanchor = {new}", flush=True)

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
todo = []
for arm, cap in FLASH:
    nm = J.fs_label(arm, cap, False, 0, "clean", 16)
    if (Path(OUT_DIR) / J.result_name(nm)).exists():
        print(f"[skip] {nm} done"); continue
    todo.append((arm, cap, nm))

def worker(gpu, jobs, failbin, packed):
    while True:
        try:
            arm, cap, nm = jobs.get_nowait()
        except queue.Empty:
            return
        if packed and cap > PACK_MAX:
            # ceiling dropped mid-wave -- reroute to the solo wave
            failbin.append((arm, cap, nm))
            continue
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True)
            continue
        log = logd / f"{arm}_g{cap}.log"
        cmd = ["python", "-u", J.FS_WORKER,
               "--data-dir", DATA_DIR, "--out-dir", OUT_DIR, "--repo", REPO,
               "--arm", arm, "--anchor-cap", str(cap),
               "--epochs", str(J.FS_EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
               "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
               "--topup-seeds", str(J.TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        print(f"[gpu{gpu}] start {nm}" + (" (packed)" if packed else ""),
              flush=True)
        t0 = time.time()
        with open(log, "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpu))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True)   # we reaped it
            if packed:
                _demote(cap)
            failbin.append((arm, cap, nm))
        print(f"[gpu{gpu}] " + ("ok" if p.returncode == 0 else "FAIL")
              + f" {nm} [{(time.time()-t0)/60:.1f} min]", flush=True)

def wave(jobs_list, per_gpu, failbin, packed):
    if not jobs_list:
        return
    jobs = queue.Queue()
    for j in jobs_list:
        jobs.put(j)
    ths = [threading.Thread(target=worker, args=(g, jobs, failbin, packed))
           for _ in range(per_gpu) for g in J.detect_gpus()]
    for t in ths:
        t.start()
    for t in ths:
        t.join()

stop_evt = threading.Event()
mon = threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True)
mon.start()
t0 = time.time()
refill, fails = [], []
wave([j for j in todo if j[1] <= PACK_MAX], PACK, refill, packed=True)
wave([j for j in todo if j[1] > PACK_MAX] + refill, 1, fails, packed=False)
stop_evt.set()
print(f"drained in {(time.time()-t0)/3600:.1f} h; {len(fails)} failed; "
      f"session maxanchor ended at {PACK_MAX}")
for _, _, nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: ZSbest-primary (val-selected zero-shot; user protocol).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_i2ce_icetf", "i2ce"),
        ("wcle_i2expce_icetf", "i2expce"),
        ("wcle_i2poolce_icetf", "i2poolce"),
        ("wcle_ceexpi2_icetf", "ceexpi2"),
        ("wcle_poolceexpi2_icetf", "poolceexpi2"),
        ("wcle_expi2expce_icetf", "expi2expce"),
        ("wcle_expi2poolexpce_icetf", "expi2poolexpce"),
        ("wcle_shexpi2ce_icetf", "shexpi2ce"),
        ("wcle_shexpi2poolce_icetf", "shexpi2poolce"),
        ("wcle_ce_cetf", "ce"),
        ("wcle_expce_cetf", "expce"),
        ("wcle_i2poolexpce_icetf", "i2poolexpce"),
        ("wcle_i2poolcmpce_icetf", "i2poolcmpce"),
        ("wcle_expi2cmpce_icetf", "expi2cmpce"),
        ("wcle_cmpi2expce_icetf", "cmpi2expce"),
        ("wcle_cmpi2cmpce_icetf", "cmpi2cmpce"),
        ("wcle_expi2poolcmpce_icetf", "expi2poolcmpce"),
        ("wcle_shexpi2poolexpce_icetf", "shexpi2poolexpce"),
        ("wcle_shcmpi2poolcmpce_icetf", "shcmpi2poolcmpce"),
        ("wcle_poolcmpce_cetf", "poolcmpce"),
        ("wcle_cecmpi2_icetf", "cecmpi2"),
        ("wcle_poolcecmpi2_icetf", "poolcecmpi2"),
        ("wcle_shcmpi2poolce_icetf", "shcmpi2poolce"),
        ("wcle_cmpi2poolexpce_icetf", "cmpi2poolexpce"),
        ("wcle_cmpi2poolcmpce_icetf", "cmpi2poolcmpce"),
        ("wcle_cmpce_cetf", "cmpce"),
        ("wcle_i2cmpce_icetf", "i2cmpce"),
        ("wcle_shcmpi2ce_icetf", "shcmpi2ce"),
        ("wcle_poolce_cetf", "poolce"),
        ("wcle_poolexpce_cetf", "poolexpce")]

def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ROWS:
    print(_row(lab, f"w9_{arm}"))


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
